# Seto-1.1B: Pretraining Session 1 (Kaggle 2x T4)

**Модель:** `Seto-1.1B` (~1,073,000,000 параметров)
- **Назначение:** Живое умное общение без цензуры + бытовой Tool Calling + размышления (`<think>`).
- **Конфигурация:** `d_model=2048`, `n_layers=22`, `n_heads=16`, `n_kv_heads=4` (GQA 4:1), `d_ff=5504`.
- **Ускоритель:** 2x Nvidia T4 (DDP, FP16, Gradient Checkpointing).
- **Токенизатор:** 48,000 токенов (BPE с токенами `<think>` и `</think>`).

In [ ]:
# Опционально: укажите ваш токен Hugging Face для автосохранения чекпоинта
# os.environ["HF_TOKEN"] = "hf_..."

import os, sys, shutil, subprocess, zipfile
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
REPO = WORK / "seto"
HF_CACHE = WORK / "hf-cache"
TOKENIZER_DIR = WORK / "seto-tokenizer"
DATA_DIR = WORK / "seto-1b-data"
OUTPUT_DIR = WORK / "seto-1b-pretrain"
SHARDS_DIR = DATA_DIR / "shards"

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TORCH_NCCL_ENABLE_MONITORING"] = "0"

def run(*command, cwd=None):
    print("+", " ".join(map(str, command)), flush=True)
    subprocess.run([str(part) for part in command], cwd=cwd, check=True)

if (REPO / ".git").exists():
    run("git", "pull", "--ff-only", cwd=REPO)
else:
    if REPO.exists():
        shutil.rmtree(REPO)
    run("git", "clone", "https://github.com/dustincorder/seto.git", REPO)

sys.path.insert(0, str(REPO))
run(sys.executable, "-m", "pip", "install", "-q", "tokenizers", "datasets", "huggingface_hub")

# 3. Проверка: возможно, шарды и токенизатор уже готовы (примонтированы из прошлого запуска)
input_shards = sorted(Path("/kaggle/input").rglob("train_0000.bin"))
if input_shards:
    SHARDS_DIR = input_shards[0].parent
    print(f"⚡ Найдена готовая папка с шардами в инпутах: {SHARDS_DIR}")
    tok_candidates = [p.parent for p in Path("/kaggle/input").rglob("tokenizer.json") if "checkpoints" not in p.parts]
    if tok_candidates:
        TOKENIZER_DIR = tok_candidates[0]
        print(f"⚡ Найден готовый токенизатор в инпутах: {TOKENIZER_DIR}")
    DATA_READY = DATA_DIR / ".ready"
    DATA_READY.touch()

DATA_READY = DATA_DIR / ".ready"
if not DATA_READY.exists() and not list(SHARDS_DIR.glob("train_*.bin")):
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("📦 Подготовка обучающих данных и токенизатора...")
    run(
        sys.executable, "scripts/prepare_data.py",
        "--output-dir", DATA_DIR,
        "--tokenizer-dir", TOKENIZER_DIR,
        "--vocab-size", "48000",
        "--max-samples-ru", "200000",
        "--max-samples-wiki", "30000",
        "--max-samples-en", "150000",
        "--max-samples-uk", "30000",
        "--max-samples-technical", "10000",
        "--shard-size", "50000000",
        cwd=REPO,
    )
    shutil.rmtree(HF_CACHE, ignore_errors=True)
    DATA_READY.touch()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("\n🚀 ЗАПУСК PRETRAINING SETO-1.1B")
run(
    "torchrun", "--standalone", "--nproc_per_node=2",
    "scripts/train.py",
    "--stage", "pretrain",
    "--model-config", "1b",
    "--data-dir", SHARDS_DIR,
    "--tokenizer", TOKENIZER_DIR,
    "--output-dir", OUTPUT_DIR,
    "--batch-size", "2",
    "--grad-accum", "8",
    "--seq-len", "1024",
    "--lr", "3e-4",
    "--min-lr", "3e-5",
    "--warmup-steps", "300",
    "--max-steps", "10000",
    "--save-every", "500",
    "--log-every", "10",
    "--fp16",
    "--gradient-checkpointing",
    cwd=REPO,
)

final_zip = OUTPUT_DIR / "final_pretrain.zip"
root_final_zip = WORK / "final_pretrain.zip"
if final_zip.exists():
    shutil.copy2(final_zip, root_final_zip)
    print(f"\n✅ Финальный архив Seto-1.1B готов: {root_final_zip} ({root_final_zip.stat().st_size / 1024**2:.1f} MB)")

try:
    run(sys.executable, "scripts/test_inference.py", "--model", root_final_zip, "--tokenizer", TOKENIZER_DIR, cwd=REPO)
except Exception as e:
    print(f"⚠️ Ошибка теста: {e}")
